# FastLp — Phase 5 A100 Accuracy-Matched Scalar Campaign

**Purpose.** Compare fixed-term Lagrange–Bürmann series, the residual-checked series/bisection fallback, and the existing Chebyshev/Newton/Bisection implementations at one empirical normalized-error target, with optional scalar objective guard overhead.

- Six exponents: $p\in\{1/3,2/5,3/4,0.9,0.99,0.999\}$.
- Same randomly generated tensor **within every exponent** for every backend. Different exponents require separate tensors.
- **Passing configurations** satisfy sampled maximum normalized error $\le 10^{-6}$, finite outputs, and zero support mismatches. Failing configurations remain in raw data and are not compared as passing speed winners.
- Timing excludes offline polynomial fitting, series coefficient precomputation, input generation and reference bisection, but these costs are recorded separately.
- Each completed exponent is verified, archived, and copied to Google Drive immediately. Interrupted Colab runtimes can restore completed cases from Drive.
- This is **not** an implementation of the original Mellin–Barnes hybrid; the fallback backend is not an interval-verified floating-point certificate. Scalar objective guard is not an inexact-FB convergence theorem.

**Before starting:** In your Mac checkout, commit and push the `revision/phase5` and `tests/test_phase5_*.py` source to branch `revision/major-experiments`. The older Phase 4 commit (`2dec97b1`) does **not** contain Phase 5. In Colab choose **Runtime → Change runtime type → A100 GPU** and then **Run all**. The notebook will stop if the Phase 5 source has not been pushed.

The first normal run is `RUN_MODE = "formal"` (1,000,000 inputs, 10 warmups and 30 timed repetitions). Choose `"pilot"` for a smaller exploratory run; its Drive checkpoint folder will be different and its results must not be conflated with formal runs.

**Pinned threshold-fix revision:** `acdef3f2902cdb67ab07006647a97dbe41f56601`. Before running any large batch, the notebook runs the full Phase 5 regression set and a CUDA support test on six exponents × three legacy backends. Results will not mix with the earlier `6930d1c7` archive.


In [ ]:
# 1. User-facing configuration (edit here only)
REPO_URL = "https://github.com/Jy-U1/Fast_Lp_Prox.git"
BRANCH = "revision/major-experiments"
# Phase 5 float32 threshold-consistency fix: pinned and pushed commit.
# Do not change mid-campaign; this SHA is also encoded in the archive ID.
EXPECTED_COMMIT = "acdef3f2902cdb67ab07006647a97dbe41f56601"

RUN_MODE = "formal"                   # "pilot" or "formal"
INCLUDE_CHEB_OBJECTIVE_GUARD = True    # scalar objective comparison only
SAVE_TO_GOOGLE_DRIVE = True            # recommended; fail early if unavailable
GOOGLE_DRIVE_PARENT = "FastLp_Phase5_A100_Archives"

assert RUN_MODE in ("pilot", "formal")
N_INPUTS = 1_000_000 if RUN_MODE == "formal" else 50_000
WARMUP = 10 if RUN_MODE == "formal" else 5
REPEATS = 30 if RUN_MODE == "formal" else 10
DTYPE = "float32"
LAMBDA = 1.0
SEED = 0
ACTIVE_FRACTION = 0.9
YMAX_FACTOR = 10.0
ACCURACY_TARGET = 1e-6
REF_STEPS = 75
HYBRID_ORDERS = "8,32,128"
HYBRID_TOL = 1e-6
HYBRID_BISECT_STEPS = 64
FIT_N = 8192
VALIDATION_N = 80001
print(f"Mode={RUN_MODE}; N={N_INPUTS:,}; repetitions={REPEATS}; scalar guard={INCLUDE_CHEB_OBJECTIVE_GUARD}")

In [ ]:
# 2. Mount persistent storage
from pathlib import Path
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_PARENT = Path("/content/drive/MyDrive") / GOOGLE_DRIVE_PARENT
    DRIVE_PARENT.mkdir(parents=True, exist_ok=True)
    assert DRIVE_PARENT.is_dir(), "Drive not mounted or not writable"
    print("Drive checkpoint parent:", DRIVE_PARENT)
else:
    DRIVE_PARENT = None
    print("WARNING: Google Drive backup is OFF. Colab disconnects can lose data.")

In [ ]:
# 3. Reproduce the pushed source and install dependencies
import os, sys, subprocess, shutil, json, hashlib, zipfile, platform
import datetime

def command(args, *, cwd=None):
    print("$", " ".join(map(str,args)), flush=True)
    subprocess.run([str(s) for s in args], cwd=cwd, check=True)

def output(args, *, cwd=None):
    return subprocess.check_output([str(s) for s in args], cwd=cwd, text=True).strip()

REPO_ROOT = Path("/content/Fast_Lp_Prox")
if not (REPO_ROOT / ".git").is_dir():
    if REPO_ROOT.exists():
        raise RuntimeError(f"{REPO_ROOT} exists but is not a git checkout")
    command(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)])
else:
    print("Reusing existing repository checkout:", REPO_ROOT)
    # Results are stored OUTSIDE the git checkout, so it should remain clean.
    assert not output(["git", "status", "--porcelain"], cwd=REPO_ROOT), "Dirty checkout; resolve changes before proceeding"
    command(["git", "fetch", "origin", BRANCH], cwd=REPO_ROOT)

if EXPECTED_COMMIT.strip():
    command(["git", "checkout", "--detach", EXPECTED_COMMIT.strip()], cwd=REPO_ROOT)
else:
    command(["git", "checkout", "--detach", f"origin/{BRANCH}"], cwd=REPO_ROOT)

COMMIT = output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT)
assert not output(["git", "status", "--porcelain"], cwd=REPO_ROOT)
if EXPECTED_COMMIT.strip():
    assert COMMIT.startswith(EXPECTED_COMMIT.strip()), "Pinned SHA mismatch"

REQUIRED = [
    REPO_ROOT / "revision/phase5/scripts/benchmark_phase5_scalar.py",
    REPO_ROOT / "revision/phase5/scripts/summarize_phase5.py",
    REPO_ROOT / "revision/phase5/backends.py",
]
missing = [str(x) for x in REQUIRED if not x.is_file()]
if missing:
    raise RuntimeError(f"Phase 5 not present at pushed commit {COMMIT}: {missing}. Commit/push Phase 5 files on your Mac first.")

command([sys.executable, "-m", "pip", "install", "-e", ".[dev]", "mpmath"], cwd=REPO_ROOT)
import torch, numpy as np, pandas as pd
assert torch.cuda.is_available(), "No CUDA device: select A100 GPU in Colab runtime settings"
GPU = torch.cuda.get_device_name(0)
TORCH_VERSION = torch.__version__
print("Frozen source SHA:", COMMIT)
print("GPU:", GPU)
print("Torch:", TORCH_VERSION, "CUDA:", torch.version.cuda)
if "A100" not in GPU.upper():
    raise RuntimeError(f"Expected an A100, found {GPU}; do not mix with archived A100 timings")


In [ ]:
# 4. Phase 5 regression + GPU threshold-consistency gate (before expensive runs)
command([sys.executable, "-m", "pytest", "-q",
    "tests/test_phase5_threshold_consistency.py",
    "tests/test_revision_bench_threshold.py",
    "tests/test_phase5_scalar.py",
    "tests/test_phase5_existing_adapter.py",
    "tests/test_phase5_reporting.py",
    "tests/test_phase5_probe_export.py",
], cwd=REPO_ROOT)

# This command is intentionally run as a module, from the repository root.
# Direct `python scripts/verify_legacy_activation.py` can fail to resolve `revision`.
command([sys.executable, "-m", "scripts.verify_legacy_activation",
    "--device", "cuda", "--n", "20000",
], cwd=REPO_ROOT)
print("Phase 5 CPU/GPU regression and CUDA support consistency checks passed")

## Six-case execution plan

The Chebyshev degrees and projected-Newton iteration counts are drawn from the earlier accuracy-matched Phase 1 settings. Extra Newton counts allow a configuration to meet the common target on the **new** shared-input tensor. Fixed-term series truncations are tested as **candidates**; it is acceptable for all fixed terms to fail for large $p$.

The scalar guard is timed as an extra `chebyshev_guarded` family, next to the unguarded Chebyshev family. It only compares the proposed scalar output with zero in floating point and is **not** the full inexact FB safeguard.

**Revision for accuracy matching.** The preceding pre-fix A100 run placed Chebyshev D12 at approximately `1.0565e-6` for p=0.75, slightly above the `1e-6` target. This notebook therefore includes D14 and D16 as well as D12, and adds higher fallback degrees for selected other exponents. Unsuccessful configurations remain reported rather than discarded.


In [ ]:
# 5. Define cases; freeze a unique, reproducible campaign ID
CASES = [
    dict(label="p03333", p="0.3333333333333333", cheb="6", newton="3,4",
         series="8,16,32,64,128"),
    dict(label="p04000", p="0.4", cheb="6,8", newton="3,4",
         series="8,16,32,64,128"),
    dict(label="p07500", p="0.75", cheb="12,14,16", newton="3,4,5",
         series="8,16,32,64,128"),
    dict(label="p09000", p="0.9", cheb="24,28", newton="3,4,5",
         series="8,16,32,64,128,256"),
    dict(label="p09900", p="0.99", cheb="54,64", newton="4,5,6",
         series="16,32,64,128,256"),
    dict(label="p09990", p="0.999", cheb="116,128", newton="4,5,6",
         series="16,32,64,128,256"),
]
assert len(CASES) == 6 and len({x["label"] for x in CASES}) == 6

SETTINGS = dict(
    source_commit=COMMIT, mode=RUN_MODE, n=N_INPUTS, dtype=DTYPE,
    gpu=GPU, torch=TORCH_VERSION, python=platform.python_version(),
    lambda_value=LAMBDA, seed=SEED, active_fraction=ACTIVE_FRACTION,
    ymax_factor=YMAX_FACTOR, accuracy_target=ACCURACY_TARGET,
    ref_steps=REF_STEPS, warmup=WARMUP, repeats=REPEATS,
    hybrid_orders=HYBRID_ORDERS, hybrid_tol=HYBRID_TOL,
    hybrid_bisect_steps=HYBRID_BISECT_STEPS,
    fit_n=FIT_N, validation_n=VALIDATION_N,
    guard_chebyshev=INCLUDE_CHEB_OBJECTIVE_GUARD,
    cases=CASES,
)
SETTINGS_HASH = hashlib.sha256(json.dumps(SETTINGS, sort_keys=True).encode()).hexdigest()[:12]
RUN_ID = f"Phase5_{RUN_MODE}_N{N_INPUTS}_{COMMIT[:8]}_{SETTINGS_HASH}"
WORK_ROOT = Path("/content/FastLp_Phase5_Work") / RUN_ID
CASE_ROOT = WORK_ROOT / "cases"
CASE_ROOT.mkdir(parents=True, exist_ok=True)
if SAVE_TO_GOOGLE_DRIVE:
    DRIVE_ROOT = DRIVE_PARENT / RUN_ID
    (DRIVE_ROOT / "case_archives").mkdir(parents=True, exist_ok=True)
    manifest_dest = DRIVE_ROOT / "settings.json"
    if manifest_dest.exists():
        stored = json.loads(manifest_dest.read_text())
        if stored != SETTINGS:
            raise RuntimeError("Existing Drive campaign manifest differs; do not mix protocols")
    else:
        manifest_dest.write_text(json.dumps(SETTINGS, indent=2, sort_keys=True) + "\n")
else:
    DRIVE_ROOT = None
print("Campaign:", RUN_ID)
print("Scratch:", WORK_ROOT)
print("Drive:", DRIVE_ROOT)


In [ ]:
# 6. Validation, SHA-256 archive creation, and safe restoration
import csv

def sha256_bytes(blob):
    return hashlib.sha256(blob).hexdigest()

def check_case(case, folder):
    fcsv = folder / "phase5_scalar_results.csv"
    fmeta = folder / "phase5_metadata.json"
    if not (fcsv.is_file() and fmeta.is_file()):
        raise FileNotFoundError(f"Incomplete case {case['label']} at {folder}")
    meta = json.loads(fmeta.read_text())
    args = meta["args"]
    expected = {
        "p": float(case["p"]), "lam": float(LAMBDA),
        "n": N_INPUTS, "dtype": DTYPE, "device": "cuda",
        "seed": SEED, "warmup": WARMUP, "repeats": REPEATS,
        "active_fraction": ACTIVE_FRACTION, "ymax_factor": YMAX_FACTOR,
        "accuracy_target": ACCURACY_TARGET, "ref_steps": REF_STEPS,
        "hybrid_orders": HYBRID_ORDERS,
        "hybrid_tol": HYBRID_TOL, "hybrid_bisect_steps": HYBRID_BISECT_STEPS,
        "series_terms": case["series"], "cheb_degrees": case["cheb"],
        "newton_iters": case["newton"], "bisect_iters": "20",
        "guard_chebyshev": INCLUDE_CHEB_OBJECTIVE_GUARD,
        "fit_n": FIT_N, "validation_n": VALIDATION_N,
        "existing_methods": "all",
    }
    for k,v in expected.items():
        if args.get(k) != v:
            raise RuntimeError(f"{case['label']}: {k} mismatch: {args.get(k)!r} != {v!r}")
    if meta["git"]["commit"] != COMMIT:
        raise RuntimeError(f"{case['label']}: source SHA mismatch")
    if meta["gpu"] != GPU or meta["torch"] != TORCH_VERSION:
        raise RuntimeError(f"{case['label']}: changed GPU/PyTorch environment")
    if len(meta.get("input_sha256", "")) != 64:
        raise RuntimeError(f"{case['label']}: missing input SHA")
    frame = pd.read_csv(fcsv)
    assert len(frame) > 0 and frame["configuration"].is_unique
    assert frame["p"].apply(lambda v: abs(float(v) - float(case['p'])) < 1e-12).all()
    assert frame["device"].eq("cuda").all() and frame["dtype"].eq(DTYPE).all()
    assert "residual_series_bisect" in set(frame["family"])
    for fam in ("fixed_series", "chebyshev", "newton", "bisection"):
        assert fam in set(frame["family"]), f"{case['label']}: missing {fam}"
    if INCLUDE_CHEB_OBJECTIVE_GUARD:
        assert "chebyshev_guarded" in set(frame["family"])
    return frame

def backup_case(case, folder):
    """Store just two source files and their digests; verified before save and after restore."""
    archive = DRIVE_ROOT / "case_archives" / (case["label"] + ".zip")
    if archive.exists():
        raise RuntimeError(f"Refusing to overwrite existing Drive case checkpoint {archive}")
    files = [folder / "phase5_scalar_results.csv", folder / "phase5_metadata.json"]
    digests = {f.name: sha256_bytes(f.read_bytes()) for f in files}
    mini = dict(case=case, source_sha=COMMIT, settings_hash=SETTINGS_HASH,
                gpu=GPU, torch=TORCH_VERSION, digests=digests)
    tmp = WORK_ROOT / (case["label"] + ".zip")
    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for f in files:
            z.write(f, f"{case['label']}/{f.name}")
        z.writestr("archive_manifest.json", json.dumps(mini, indent=2))
    with zipfile.ZipFile(tmp) as z:
        assert z.testzip() is None
    shutil.copy2(tmp, archive)
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None, f"Drive archive corrupted: {archive}"
    print("Saved Drive checkpoint:", archive, flush=True)

def restore_case(case, folder):
    archive = DRIVE_ROOT / "case_archives" / (case["label"] + ".zip")
    if not archive.is_file():
        return False
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None, f"Drive checkpoint corrupt: {archive}"
        manifest = json.loads(z.read("archive_manifest.json"))
        if not (manifest["case"] == case and manifest["source_sha"] == COMMIT
                and manifest["settings_hash"] == SETTINGS_HASH
                and manifest["gpu"] == GPU and manifest["torch"] == TORCH_VERSION):
            raise RuntimeError(f"Checkpoint metadata mismatch for {case['label']}")
        folder.mkdir(parents=True, exist_ok=True)
        for fname in ("phase5_scalar_results.csv", "phase5_metadata.json"):
            raw = z.read(f"{case['label']}/{fname}")
            if sha256_bytes(raw) != manifest["digests"][fname]:
                raise RuntimeError(f"Checkpoint SHA256 mismatch: {case['label']}/{fname}")
            (folder / fname).write_bytes(raw)
    check_case(case, folder)
    print("Restored verified case:", case["label"], flush=True)
    return True


In [ ]:
# 7. Run six A100 cases. Each successful case is immediately checkpointed to Drive.
# If interrupted, run the whole notebook again; checkpointed cases are verified and skipped.
import time

for i, case in enumerate(CASES, 1):
    folder = CASE_ROOT / case["label"]
    print("\n" + "=" * 72)
    print(f"Case {i}/{len(CASES)}: p={case['p']} ({case['label']})", flush=True)

    if folder.exists():
        check_case(case, folder)
        print("Already verified in this session:", case["label"])
        if SAVE_TO_GOOGLE_DRIVE:
            archive = DRIVE_ROOT / "case_archives" / (case["label"] + ".zip")
            if not archive.exists():
                backup_case(case, folder)
        continue
    if SAVE_TO_GOOGLE_DRIVE and restore_case(case, folder):
        continue

    argv = [sys.executable, "-m", "revision.phase5.scripts.benchmark_phase5_scalar",
        "--device", "cuda", "--dtype", DTYPE,
        "--p", case["p"], "--lam", str(LAMBDA),
        "--n", str(N_INPUTS), "--seed", str(SEED),
        "--active-fraction", str(ACTIVE_FRACTION),
        "--ymax-factor", str(YMAX_FACTOR),
        "--accuracy-target", str(ACCURACY_TARGET),
        "--ref-steps", str(REF_STEPS),
        "--warmup", str(WARMUP), "--repeats", str(REPEATS),
        "--series-terms", case["series"],
        "--hybrid-orders", HYBRID_ORDERS,
        "--hybrid-tol", str(HYBRID_TOL),
        "--hybrid-bisect-steps", str(HYBRID_BISECT_STEPS),
        "--existing-methods", "all",
        "--cheb-degrees", case["cheb"],
        "--newton-iters", case["newton"],
        "--bisect-iters", "20",
        "--fit-n", str(FIT_N),
        "--validation-n", str(VALIDATION_N),
        "--output-dir", str(folder),
    ]
    if INCLUDE_CHEB_OBJECTIVE_GUARD:
        argv.append("--guard-chebyshev")

    t0 = time.time()
    command(argv, cwd=REPO_ROOT)
    frame = check_case(case, folder)
    print(f"Verified {len(frame)} method configurations in {time.time()-t0:.1f} wall seconds")
    if SAVE_TO_GOOGLE_DRIVE:
        backup_case(case, folder)

print("All six Phase 5 scalar experiments produced validated case results.")


In [ ]:
# 8. Check accuracy-matched winners and collect raw results
CSV_FILES = [CASE_ROOT / x["label"] / "phase5_scalar_results.csv" for x in CASES]
for case in CASES:
    check_case(case, CASE_ROOT / case["label"])

SUMMARY_ROOT = WORK_ROOT / "summary"
if SUMMARY_ROOT.exists():
    # Owned generated folder: recreate summary to reflect all verified cases.
    shutil.rmtree(SUMMARY_ROOT)
command([sys.executable, "-m", "revision.phase5.scripts.summarize_phase5",
         *[str(x) for x in CSV_FILES], "--output-dir", str(SUMMARY_ROOT)], cwd=REPO_ROOT)

raw_blocks=[]
for case in CASES:
    f = CASE_ROOT / case["label"] / "phase5_scalar_results.csv"
    block = pd.read_csv(f)
    block.insert(0, "case", case["label"])
    raw_blocks.append(block)
all_raw = pd.concat(raw_blocks, ignore_index=True)
all_raw.to_csv(SUMMARY_ROOT / "all6_raw_results.csv", index=False)
selected = pd.read_csv(SUMMARY_ROOT / "phase5_matched_summary.csv")
print("Total raw backend configurations:", len(all_raw))
print("PASS rate:", int(all_raw["meets_accuracy_target"].sum()), "/", len(all_raw))
print("Passing family-by-p options:", int((selected["status"]=="PASS").sum()), "/",len(selected))
from IPython.display import display
display(selected[["p", "family", "status", "configuration", "median_ms", "max_norm_error", "fallback_count"]])


In [ ]:
# 9. Final verified ZIP of ALL original raw outputs + audit + environment info
from datetime import datetime, timezone

FINAL_NAME = (f"FastLp_Phase5_A100_{RUN_MODE}_All6_{COMMIT[:8]}_"
              f"{datetime.now(timezone.utc):%Y%m%d_%H%M%S}.zip")
FINAL_ZIP = Path("/content") / FINAL_NAME

RUN_MANIFEST = dict(
    generated_utc=datetime.now(timezone.utc).isoformat(),
    campaign=RUN_ID, run_settings=SETTINGS, git_commit=COMMIT,
    git_clean=(output(["git", "status", "--porcelain"], cwd=REPO_ROOT)==""),
    gpu=GPU, torch=TORCH_VERSION, cuda=torch.version.cuda,
    python=platform.python_version(),
    case_count=len(CASES), raw_backend_rows=len(all_raw),
    all_cases_verified=True,
    note="GPU timings are Phase-5 shared-input trials; NOT the previous Phase-1 GPU input distribution.",
    limits="Residual-checked bisection fallback is not the published Mellin-Barnes hybrid or a floating-point certificate. Objectiveguard tests the scalar objective, not an inexact FB proof.",
)
manifest_file = WORK_ROOT / "final_manifest.json"
manifest_file.write_text(json.dumps(RUN_MANIFEST, indent=2, sort_keys=True) + "\n")

with zipfile.ZipFile(FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for case in CASES:
        folder = CASE_ROOT / case["label"]
        for fname in ("phase5_scalar_results.csv", "phase5_metadata.json"):
            f = folder / fname
            z.write(f, f"cases/{case['label']}/{fname}")
    for f in sorted(SUMMARY_ROOT.rglob("*")):
        if f.is_file():
            z.write(f, f"summary/{f.relative_to(SUMMARY_ROOT)}")
    z.write(manifest_file, "final_manifest.json")

with zipfile.ZipFile(FINAL_ZIP) as z:
    bad = z.testzip()
    assert bad is None, f"ZIP CRC verification failed at {bad}"
    names = z.namelist()
    assert sum(name.endswith("/phase5_scalar_results.csv") for name in names)==6

print("Complete archive:", FINAL_ZIP)
print("Archive size MB:", round(FINAL_ZIP.stat().st_size/1024**2, 3))
print("Verified ZIP entries:",len(names))
if SAVE_TO_GOOGLE_DRIVE:
    dest = DRIVE_ROOT / FINAL_NAME
    shutil.copy2(FINAL_ZIP, dest)
    assert dest.stat().st_size == FINAL_ZIP.stat().st_size
    with zipfile.ZipFile(dest) as z:
        assert z.testzip() is None
    print("Persistent Drive copy:",dest)

from google.colab import files
files.download(str(FINAL_ZIP))
print("Upload the COMPLETE ZIP to ChatGPT for numerical audit and manuscript response.")


## After downloading

The final archive contains each of the six original `phase5_scalar_results.csv` and `phase5_metadata.json` pairs, selected passing configurations, the combined raw CSV, source commit and environment manifest. **Retain both passing and failing rows.**

Suggested paper interpretation: report the best **passing** configuration per family and exponent, and separately report offline build time, actual fallback fraction, and the scalar guard overhead. Neither the original [24] Mellin–Barnes MATLAB baseline nor the complete inexact FB safeguard has been evaluated by this notebook. Do not claim those tasks were completed.